***
# Homework 9: SQL and APIs

**Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** April 8th, 2024
***

In [71]:
import os
import sqlite3
import requests

## 1.) Warmup: `sqlite3` (3 points, spent $\approx$ 5 minutes)

**Here is a table similar to the ones that we saw in the lecture slides, describing some information about the colleges in the West Division of the Big 10 conference.**

![](image.png)

**Use `sqlite3` to create a database with a single table (in addition to the standard metainformation tables) called `t_big10west` that recreates the table in the figure above. That is, `t_big10west` should have five columns `(ID, University, City, State, Founded)`, and seven rows corresponding to the seven universities in the table. Save the database in a file called `big10.db`, and include this file in your submission.**

In [72]:
data = [(101, 'University of Illinois', 'Urbana', 'Illinois', 1867),
        (202, 'University of Iowa', 'Iowa City', 'Iowa', 1847),
        (303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851),
        (404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869),
        (505, 'Northwestern University', 'Evanston', 'Illinois', 1851),
        (606, 'Purdue University', 'West Lafayette', 'Indiana', 1869),
        (707 , 'University of Wisconsin', 'Madison', 'Wisconsin', 1849)]

# If big10.db already exists, we'll get yelled at when we
# try to create a DB file now, so delete it if it
# already exists.
UNIV_DB_FILE = 'big10.db'
if os.path.exists( UNIV_DB_FILE ):
    os.remove( UNIV_DB_FILE )

# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

# Create a table
cursor.execute('''
                CREATE TABLE t_big10west ('ID', 
                                          'University', 
                                          'City', 
                                          'State', 
                                          'Founded') 
                ''')

# Insert data into the table
cursor.executemany('''
                    INSERT INTO t_big10west
                    VALUES (?, ?, ?, ?, ?) 
                    ''', data)

# Commit the transaction
conn.commit()

# Close the transaction
conn.close()

In [73]:
# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

for row in cursor.execute(''' 
                          SELECT * FROM t_big10west 
                          '''):
        print(row)

# Close the transaction
conn.close()

(101, 'University of Illinois', 'Urbana', 'Illinois', 1867)
(202, 'University of Iowa', 'Iowa City', 'Iowa', 1847)
(303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851)
(404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869)
(505, 'Northwestern University', 'Evanston', 'Illinois', 1851)
(606, 'Purdue University', 'West Lafayette', 'Indiana', 1869)
(707, 'University of Wisconsin', 'Madison', 'Wisconsin', 1849)


**Oops! There’s a typo in that table. The University of Wisconsin was founded in 1848, not 1849. Write a SQL command to correct the corresponding entry of the table, and save it in a string-valued variable called `big10_correction`. You do not need to run this command (but you probably should, to check that it’s correct, and if you do, and you change the entry in the table in `big10.db`, that’s okay)**

In [74]:
# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

cursor.execute(''' 
               UPDATE t_big10west 
               SET Founded = 1848
               WHERE ID = 707 AND University = 'University of Wisconsin' AND City = 'Madison' AND State = 'Wisconsin'
               ''')

for row in cursor.execute('''
                          SELECT * FROM t_big10west
                          '''):
    print(row)

# Close the transaction
conn.close()

(101, 'University of Illinois', 'Urbana', 'Illinois', 1867)
(202, 'University of Iowa', 'Iowa City', 'Iowa', 1847)
(303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851)
(404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869)
(505, 'Northwestern University', 'Evanston', 'Illinois', 1851)
(606, 'Purdue University', 'West Lafayette', 'Indiana', 1869)
(707, 'University of Wisconsin', 'Madison', 'Wisconsin', 1848)


## 2.) Relational Databases and SQL (7 points, spent $\approx$ 25 minutes)

**In this problem, you’ll interact with a toy SQL database using Python’s built-in sqlite3 package. Documentation can be found at** 
<h5 align="center"> https://docs.python.org/3/library/sqlite3.html </h5>

**For this problem, we’ll use a popular toy SQLite database, called `Chinook`, which represents a digital music collection. See the documentation at:**
<h5 align="center"> https://github.com/lerocha/chinook-database/blob/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite </h5>

**or a more detailed explanation. We’ll use the `.sqlite` file `Chinook_Sqlite.sqlite`, which you should download from the GitHub page above.** 

**Note: Don’t forget to save the file in the directory that you’re going to compress and hand in, and make sure that you use a relative path when referring to the file, so that when the grading script runs your code on one of our machines the file path will still work!**

**Load the database using the Python `sqlite3` package. How many tables are in the database? Save the answer in the variable `n_tables`.**

In [75]:
Chinook = 'Chinook_Sqlite.sqlite'
conn = sqlite3.connect( Chinook )
cursor = conn.cursor()

n_tables = 0
for table in cursor.execute('''
                            SELECT *
                            FROM sqlite_master
                            '''):
    if table[0] == 'table':
        n_tables += 1
print(f'There are {n_tables} tables in {Chinook}.')

conn.close()

There are 11 tables in Chinook_Sqlite.sqlite.


**What are the names of the tables in the database? Save the answer as a list of strings, `table_names`.** 

**Note: you should write Python `sqlite3` code to answer this; don’t just look up the answer in the documentation!**

In [76]:
Chinook = 'Chinook_Sqlite.sqlite'
conn = sqlite3.connect( Chinook )
cursor = conn.cursor()

table_names = []
for table in cursor.execute('''
                            SELECT *
                            FROM sqlite_master
                            WHERE type = 'table'
                            '''):
    table_names.append(table[1])
    
print(f'The table names in {Chinook} are:')
print(table_names)

conn.close()

The table names in Chinook_Sqlite.sqlite are:
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


**Write a function `list_album_ids_by_letter` that takes as an argument a single character (i.e., a string of length one) and returns a list of the primary keys of all the albums whose titles start with that character.** 

**Your function should ignore case, so that the inputs `“a”` and `“A”` yield the same results.** 

**Include error checking that raises an appropriate error in the event that the input is of the wrong type or if it is not a single character.**

In [77]:
def get_column_names(table_name, cursor):
    
    cursor.execute(f'''
                   SELECT * 
                   FROM {table_name}
                   ''')

    cursor.fetchall()

    columns = []
    for col in cursor.description:
        columns.append(col[0])
    
    return columns

In [78]:
def list_album_ids_by_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    
    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    AlbumID_idx = get_column_names('Album', cursor).index('AlbumId')
    Title_idx = get_column_names('Album', cursor).index('Title')

    result = []
    for table in cursor.execute('''
                                SELECT *
                                FROM Album
                                '''):
        album_title = table[Title_idx]
        if album_title[0] == char:
            result.append(table[AlbumID_idx])
            
    conn.close()

    return result

print(list_album_ids_by_letter('a'))

[10, 14, 15, 24, 26, 29, 74, 75, 85, 89, 90, 94, 95, 96, 120, 139, 160, 167, 168, 169, 203, 224, 232, 233, 248, 254, 272, 273, 285, 296, 307, 319]


**Write a function `list_song_ids_by_album_letter` that takes as an argument a single character and returns a list of the primary keys of all the songs whose album names begin with that letter (again ignoring case).** 

**As in `list_album_ids_by_letter`, your function should ignore case and perform error checking as appropriate.**

**Hint: you’ll need a JOIN statement here. You can use the `cursor.description` attribute to find out about tables and the names of their columns.**

In [79]:
def list_song_ids_by_album_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    album_ids = list_album_ids_by_letter(char)
    
    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    TrackId_idx = get_column_names('Track', cursor).index('TrackId')
    AlbumId_idx = get_column_names('Track', cursor).index('AlbumId')
    
    result = []
    for table in cursor.execute('''
                                SELECT *
                                FROM Track
                                '''):
        if table[AlbumId_idx] in album_ids:
            result.append(table[TrackId_idx])

    conn.close()

    return result

print(list_song_ids_by_album_letter('a'))

[85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157, 1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210, 1211, 1212, 1213, 1214, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1223, 1224, 1225, 1226, 1227, 1228, 1229, 1230, 1231, 1232, 1233, 1234, 1479, 1480, 148

**Write a function `total_cost_by_album_letter` that takes as an argument a single character and returns the total cost of buying all the songs whose album begins with that letter.**

**This cost should be based on the tracks’ unit prices, so that the cost of buying a set of tracks is simply the sum of the unit prices of all the tracks in the set.**

**Again your function should ignore case and perform appropriate error checking.**

In [80]:
def total_cost_by_album_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    song_ids = list_song_ids_by_album_letter(char)

    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    TrackId_idx = get_column_names('Track', cursor).index('TrackId')
    UnitPrice_idx = get_column_names('Track', cursor).index('UnitPrice')

    summation = 0
    for table in cursor.execute('''
                                SELECT *
                                FROM Track
                                '''):
        if table[TrackId_idx] in song_ids:
            summation += table[UnitPrice_idx]

    conn.close()

    return summation

print(total_cost_by_album_letter('a'))

366.3100000000019


## 3.) Warmup: interacting with the Yelp API (5 points, spent $\approx$ 15 minutes)

**In this problem, you’ll get some practice working with the Yelp API, which we already saw in lecture.**

**First, you need to obtain an API key in order to authenticate to the Yelp API. Follow the instructions at**
<h5 align="center"> https://docs.developer.yelp.com/docs/fusion-authentication </h5>

**under the section titled “Create an app on Yelp’s Developers site”. You may fill in whatever information you like in the app information.** 

**Note that you will need a Yelp account to create an app, which you need in order to obtain an API key. If you do not feel comfortable doing this, please let me know promptly by email.**

**Once you have filled out your information, you will be given a ClientID and an API Key. This ID and key come with an associated 300 free calls to the Yelp API to use in the month following the day you create your app.** 

**You should not need anywhere near these 300 API calls to test your code, but if you do run out of API calls, please let me know promptly.**

In [81]:
# Successfully obtained API key to authenticate Yelp API.
yelp_api_key = 'UqnvT2UCj3Jj_6DG62kXYvZWwAZU2xISaFddyUjB5XmHokDBIbZxJOjlcLZYntm2D4OO8b82yzvreyLm1vme7cEt-V2BaOoDsAD36NLQ3VyhI445LZv2zgcWVFAXZnYx'
yelp_client_id = 'LCPZKyKBQpu91swB9LRc1g'

**Write a function called `near_msc` that takes three arguments: a string, a non-negative integer and another string, in that order, representing a search string, a distance in meters and a Yelp API key, respectively.**

**`near_msc( s, d, key )` should return a list of strings, representing the Yelp aliases of all of the establishments matching the given search string s that are within d meters meters of the statistics department (1300 University Ave, Madison WI), using the given API key.**

**The distance argument should default to 1000. You may have the `key` argument default however you want. Note that we are including this optional `key` argument so that when it comes time to test your code, we can swap out your API key for that of the instructor or the grader.** 

**It will be most convenient for you to have this argument default to your API key, but be sure to change this behavior before submitting the assignment if you do not wish to share your API key with the instructor and grader (we will use our own keys to test your code, anyway, of course).**

**Your function should perform error checking to ensure that the arguments are of the right type, and you should raise an appropriate error in the event that the distance argument is negative.** 

**Hint: you have my permission to modify the code from the slides, which already essentially carries out this operation.** 

**Second hint: see the documentation at**
<h5 align="center"> https://docs.developer.yelp.com/reference/v3_business_search </h5>

In [82]:
def near_msc(s, d = 1000, key = yelp_api_key):

    if not isinstance(s, (str, )):
        raise TypeError(f's ({s}) must be of type str.')
    
    if not isinstance(d, (int, float, )):
        raise TypeError(f'd ({d}) must be of type int or float.')
    if d < 0:
        raise ValueError(f'd ({d}) must be non-negative.')
    
    if not isinstance(key, (str, )):
        raise TypeError(f'key ({key}) must be of type str.')
    
    url = 'https://api.yelp.com/v3/businesses/search'
    headers = {'Authorization': f'Bearer {key}'}
    url_params = {'term': f'{s}',
                  'radius': f'{d}',
                  'location': '1300 University Ave, Madison WI'}
    
    r = requests.get(url, headers = headers, params = url_params)
    return [res['alias'] for res in r.json()['businesses']]

In [83]:
near_msc('restaurant', 1000)

['sweet-home-wisconsin-madison',
 'chopsticks-no-title-3',
 'fabiolas-spaghetti-house-and-deli-madison',
 'camp-cantina-no-title',
 'kosharie-madison',
 'qq-express-madison',
 'butterbird-madison',
 'jordans-big-ten-pub-madison',
 'the-library-cafe-and-bar-madison',
 'steenbocks-on-orchard-madison',
 'nams-noodle-and-karaoke-bar-madison-2',
 'asian-noodle-madison',
 'saigon-sandwich-madison-madison',
 'hong-kong-station-madison-3',
 'sconniebar-madison',
 'maries-soul-food-madison',
 'the-sett-madison',
 'mickies-dairy-bar-madison',
 'hong-kong-cafe-madison',
 'aldos-cafe-madison']

**Write a function called `best_near_msc` that has the same signature as `near_msc` (i.e., takes the same arguments and has the same default behavior) and returns a string representing the alias of the highest-rated establishment matching the given search string and within the given distance of the statistics department.** 

**If no businesses exist inside the given distance, your function should return `None`.** 

**Note that the ratings of the businesses are rounded to the nearest half star, so you will likely have ties, which you may break arbitrarily.** 

**Hint: it will be easiest to retrieve some search results and look at the attributes of the resulting JSON objects. You’re looking for an attribute that corresponds to a rating.**

In [84]:
def best_near_msc(s, d = 1000, key = yelp_api_key):
    
    if not isinstance(s, (str, )):
        raise TypeError(f's ({s}) must be of type str.')
    
    if not isinstance(d, (int, )):
        raise TypeError(f'd ({d}) must be of type int.')
    if d < 0:
        raise ValueError(f'd ({d}) must be non-negative.')
    
    if not isinstance(key, (str, )):
        raise TypeError(f'key ({key}) must be of type str.')
    
    url = 'https://api.yelp.com/v3/businesses/search'
    headers = {'Authorization': f'Bearer {key}'}
    url_params = {'term': f'{s}',
                  'radius': f'{d}',
                  'location': '1300 University Ave, Madison WI'}
    
    r = requests.get(url, headers = headers, params = url_params)
    temp = r.json()

    if len(temp['businesses']) == 0:
        return None
    
    minimum = 0
    index = 0

    for i, res in enumerate(temp['businesses']):

        if res['rating'] > minimum:
            alias = res['alias']
            minimum = res['rating']
            index = i
            
        elif res['rating'] == minimum:
            if res['review_count'] > temp['businesses'][index]['review_count']:
                alias = res['alias']
                minimum = res['rating']
                index = i

    return alias

In [85]:
best_near_msc('restaurant', 1000)

'chopsticks-no-title-3'

## 4.) Tracking Asteroids with NASA’s NeoWs API (10 points, spent $\approx$ 50 minutes)

**In this problem, you’ll get more practice working with APIs, this time using one maintained by NASA for retrieving information about near earth objects (NEOs), asteroids that pass close to Earth. The documentation is available at**

<h5 align="center"> https://api.nasa.gov/ </h5>

**(scroll down to the API titled *Asteroids NeoWs*).**

**First and foremost, you’ll need an API key for accessing the service. You can get one at**

<h5 align="center"> https://api.nasa.gov/ </h5>

**You’ll need to supply an email address, which can be either your Wisconsin email or a personal email address.**

In [86]:
# Successfully obtained Nasa API Key
nasa_api_key = 'BZeQmerI1U4TenI9frtwTWNsrPY1TNqyL3ZEhPC2'

**We’ll use the Asteroids NeoWs Feed to retrieve Near Earth Objects based on the date of their closest approach to Earth. This can be done using the Feed service. If you read the documentation, you’ll see that the Feed API is accessible at**

<h5 align="center"> https://api.nasa.gov/neo/rest/v1/feed </h5>

**and takes three URL parameters: `start_date`, `end_date` and `api_key`.** 

* **`start_date` and `end_date` specify the start and end of a date range, both formatted as `YYYY-MM-DD`.**

* **`api_key` specifies the API key that you requested previously.**

**Retrieve a JSON object from the NASA NeoWs Feed API for January 1st, 2015 (i.e., set `start_date` and `end_date` to be ’2015-01-01’). You’ll notice that the JSON object has three attributes:**

* **`element_count`: the number of near earth objects that had their nearest approach during the time spanned by `start_date` and `end_date`.**

* **`near_earth_objects`: a JSON object whose attributes are the dates (represented by strings of the form `YYYY-MM-DD`) in the time spanned by `start_date` and `end_date`. Each such date attribute has as its value an array of JSON objects, each of which represents a near Earth object.**

* **`links`: URLs pointing to the “current” day, and the days before and after**

**The JSON object for January 1st, 2015 should have `element_count` attribute equal to 14. That is, if your JSON object is stored in `neo_json`, evaluating `neo_json['element_count']` should be return 14.** 

**JSON objects representing the NEOs are stored in the array `neo_json['near_earth_objects']['2015-01-01']`**

**If you pick out one of the JSON objects in this array, it should have attributes that include strings like `'estimated_diameter'` and `'is_potentially_hazardous_asteroid'`**

**Extract the names of all of these attributes and store them in a Python list called `neo_attrs`.**

In [87]:
def Access_NASA_API(start_date, end_date, key = nasa_api_key):

    if not isinstance(start_date, (str, )):
        raise TypeError(f'start_date ({start_date}) must be of type str.')
    
    if not isinstance(end_date, (str, )):
        raise TypeError(f'end_date ({end_date}) must be of type int or float.')
    
    if not isinstance(key, (str, )):
        raise TypeError(f'key ({key}) must be of type str.')
    
    url = 'https://api.nasa.gov/neo/rest/v1/feed'
    headers = {'Authorization': f'Bearer {key}'}
    url_params = {'start_date': f'{start_date}',
                  'end_date': f'{end_date}',
                  'api_key': f'{key}'}
    
    r = requests.get(url, headers = headers, params = url_params)
    neo_json = r.json()
    
    return neo_json

neo_json = Access_NASA_API('2015-01-01', '2015-01-01')
neo_attrs = list(neo_json['near_earth_objects']['2015-01-01'][0].keys())
print(neo_attrs)

['links', 'id', 'neo_reference_id', 'name', 'nasa_jpl_url', 'absolute_magnitude_h', 'estimated_diameter', 'is_potentially_hazardous_asteroid', 'close_approach_data', 'is_sentry_object']


**We’re going to write a function to retrieve all the near Earth objects from a particular day.** 

**To start, write a function called `get_neos_response` that takes three positive integers, `yyyy`, `mm` and `dd`, and a string key, in that order. `yyyy`, `mm` and `dd` will encode a year, a month and a date, respectively, and the string key will be an API key.** 

**`get_neos_response` should return the JSON object that is returned by the NASA NeoWs Feed for the specified day. `yyyy` should be a required argument, but `mm` and `dd` should be optional, with both defaulting to 1.** 

**The `key` argument should also be optional, and you may have it default however you like (see our discussion in Problem 1 regarding leaving your API key in the code). You may assume that the first three arguments are all of the appropriate type (i.e., integers), that they are all positive and that `yyyy-mm-dd` specifies a valid date.** 

**That is, the grader script will not try to do something funny like get the objects from October 32 by calling `get_neos_response(2023,10,32)` or get a day in the year 0 or -70, or get the objects from February 29 in a non-leap year. You also do not need to worry about the fact that this data set only goes goes back to about 1900.** 

**Hint: Break the problem down into simple steps:**

* **(a.) Use the supplied year, month and day to construct the `start_date` and `end_date` string arguments (which should be the same!). Hint: you may want to write a helper function to do this.**

* **(b.) Use `start_date` and `end_date` along with the API key to create a dictionary of URL parameters, and use the Python requests module to submit an HTTPGET request to the NASA NeoWs Feed API.**

* **(c.) Extract the JSON object stored in the resulting requests object and return it.**

**Note: your function should return a JSON object (i.e., a Python dictionary), not its string representation. The JSON object your return should have attributes `'links'`, `'near_earth_objects'` and `'element_count'`.**

In [88]:
def get_neos_response(yyyy, mm, dd, key = nasa_api_key):
        
    start_date = f'{yyyy}-{mm}-{dd}'
    end_date = f'{yyyy}-{mm}-{dd}'

    neo_json = Access_NASA_API(start_date, end_date, key)
    
    return neo_json['near_earth_objects']

**So we’re in the process of writing a function to retrieve the near Earth objects from a given day. Before moving on, though, it will be useful to have a function for checking that the year, month and date supplied by the user actually encode a real date.** 

**Write a function `is_valid_date` that takes three integer arguments, `yyyy`, `mm` and `dd`, in that order, and returns a Boolean that is true if and only if the arguments specify a valid date (that is, a date that actually happens in the Gregorian calendar).** 

**For example:**

* **`yyyy = 1900`, `mm = 13`, `dd = 2` is invalid, because there is no thirteenth month.** 

* **`yyyy = 2020`, `mm = 10`, `dd = 32` is invalid because there is no October 32nd.**

* **`yyyy = 2020`, `mm = 0`, `dd = 1` is invalid because there is no month zero.** 

**Your function should return False if any of the arguments are not positive. You need not perform any error checking in this function. That is, there is no need to check that the arguments are integers– we will do that “upstream” in our code, before passing arguments into `is_valid_date`.** 

**Hint: you may find it useful to create a dictionary that maps the numbers 1 through 12 to the number of days in the corresponding months (i.e., 1 maps to 31, the number of days in January, 2 maps to 28, the number of days in February during a non-leap year, 3 maps to 31, the number of days in March, etc.).**

In [89]:
def is_valid_date(yyyy, mm, dd):

    leap = False
    if (yyyy % 4 == 0) and (yyyy % 100 != 0) or (yyyy % 400 == 0):
        leap = True

    if mm < 1 or mm > 12:
        return False

    if dd < 1 or dd > 31:
        return False

    if not leap:
        valid_days_in_month = {1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30, 7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31}
    else:
        valid_days_in_month = {1: 31, 2: 29, 3: 31, 4: 30, 5: 31, 6: 30, 7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31} 

    if dd > valid_days_in_month[mm]:
        return False

    return True

assert( is_valid_date(2015, 1, 1) == True )  
assert( is_valid_date(2015, 2, 29) == False )
assert( is_valid_date(2019, 2, 29) == False )
assert( is_valid_date(1900, 13, 2) == False )
assert( is_valid_date(2000, 2, 29) == True )
assert( is_valid_date(2001, 10, 32) == False )

**Our function get_neos_response gets us a JSON object, but we’re really just interested in the asteroids, not the extra information included in the JSON object.** 

**Write a function get_neos that has the same signature as `get_neos_response` (i.e., takes the same arguments and has the same default behavior), in which the three integer arguments specify a date and the key argument specifies an API key, and returns a list of JSON objects that represent the near Earth objects that made their closest approach on the specified date.** 

**Your function should check that the arguments are of the appropriate type and that they are all positive, and raise an appropriate error if they are not. Your function should use `is_valid_date` to check that the arguments jointly describe a valid date and raise an appropriate error if they do not.**

**You do not need to worry about the fact that this data set only goes back to about 1900. You should raise an error if any of the three integer arguments are not positive, but otherwise, so long as the arguments specify a valid date, there is no need to raise an error, even though the specified date might not have any data, or might even be a day in the future!** 

**Indeed, if you write this code in a reasonable way, it will happen automatically that if the user specifies a valid date that has no data associated to it, your code will just return an empty list.** 

**Hint: once again, you may find it helpful to break the problem down into simpler steps:**

* **Perform error checking.**

* **Use `get_neos_response` to get the JSON object for the given date from the NASA API.**

* **Extract the array of near Earth objects.**

* **Return that array as a Python list.**

In [90]:
def get_neos(yyyy, mm, dd, key = nasa_api_key):

    if not isinstance(yyyy, (int, )):
        raise TypeError(f'yyyy ({yyyy}) must be of type int.')
    
    if not isinstance(mm, (int, )):
        raise TypeError(f'mm ({mm}) must be of type int.')
    
    if not isinstance(dd, (int, )):
        raise TypeError(f'dd ({dd}) must be of type int.')
    
    if yyyy < 0:
        raise ValueError(f'yyyy ({yyyy}) must be non-negative.')
    if mm < 0:
        raise ValueError(f'mm ({mm}) must be non-negative.')
    if dd < 0:
        raise ValueError(f'dd ({dd}) must be non-negative.')
    
    if not isinstance(key, (str, )):
        raise TypeError(f'key ({key}) must be of type str.')
    
    if not is_valid_date(yyyy, mm, dd):
        raise ValueError(f'The date {yyyy}-{mm}-{dd} is not valid.')
    
    if dd < 10:
        dd = f'0{dd}'
    if mm < 10:
        mm = f'0{mm}'

    neo_json = get_neos_response(yyyy, mm, dd, key)

    if len(neo_json) == 0:
        return None
    else:
        return neo_json[f'{yyyy}-{mm}-{dd}']

**Each near Earth object that we get from the API has a number of attributes describing the asteroid. Among these is the `'estimated_diameter'` attribute, whose value is another JSON object that gives the maximum and minimum (estimated) diameter of the asteroid in several different units (e.g., miles, kilometers, etc.).** 

**Write a function `get_neos_avg_maxdiam_km` that has the same signature as `get_neos_response` and `get_neos` and the same default values, and returns the average maximum diameter in kilometers of all the near Earth objects that made their closest approach on the given day.** 

**If no near earth objects made their nearest approach on the given day, your function should return `None`.** 

**Your function should perform error checking as described in `get_neos`, but if you’re careful, you won’t need to write any error checking in this function— `get_neos` already does error checking for us!**

In [92]:
def get_neos_avg_maxdiam_km(yyyy, mm, dd, key = nasa_api_key):

    neo_list = get_neos(yyyy, mm, dd, key)

    if neo_list is None:
        return None
    
    else:
        summation = 0
        for i in range(len(neo_list)):
            summation += neo_list[i]['estimated_diameter']['kilometers']['estimated_diameter_max']
        return summation / len(neo_list)

get_neos_avg_maxdiam_km(2022, 11, 6)

0.19039598156774198

**One thing we might like to explore is how the number of near Earth objects per day changes from day to day. Is this number correlated from one day to the next? Does it vary seasonally (i.e., does it have a time-varying component)?** 

**Write a function `count_neos` that has the same signature as `get_neos_response` and `get_neos` and returns a nonnegative integer corresponding to the number of near Earth objects that made their closest approach on the given day.** 

**Your function should perform error checking as described in get_neos, but once again, you should be able to rely on `get_neos` to do that for you.**

In [93]:
def count_neos(yyyy, mm, dd, key = nasa_api_key):

    neo_list = get_neos(yyyy, mm, dd, key)

    if neo_list is None:
        return 0
    else:
        return len(neo_list)

***Bonus (not worth any points, just bragging rights):***

**Use your new-found knowledge of this API to find the object discussed in this news story from last year (and which was the inspiration for this homework problem), noting that the API does indeed support retrieving closest approach data for dates in the future:**
<h5 align="center"> https://www.npr.org/2021/03/27/981917655 </h5>

The following quote from the above news story helps us locate the object discussed in the news story:

> Though the risk for impact in **2029** was ruled out long ago, Apophis will come within 20,000 miles of Earth's surface that year on **April 13**. Observers in the Eastern hemisphere will have a chance to see it without binoculars, and astronomers have an opportunity to learn more about the asteroid, without the worry that it is still a risk to the planet.

Therefore, to locate Apophis, we do the following:

* We use `get_neos()` to find all near-earth objects when `yyyy = 2029`, `mm = 4`, and `dd = 13`.

* We run a for-loop that goes through every object till it finds the name with "Apophis" in it.

* Print the output.

In [103]:
near_objects = get_neos(2029, 4, 13)

for i in range(len(near_objects)):
    split_name = near_objects[i]['name'].split()
    if 'Apophis' in split_name:
        object_located = near_objects[i]
        break

object_located

{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/2099942?api_key=BZeQmerI1U4TenI9frtwTWNsrPY1TNqyL3ZEhPC2'},
 'id': '2099942',
 'neo_reference_id': '2099942',
 'name': '99942 Apophis (2004 MN4)',
 'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2099942',
 'absolute_magnitude_h': 19.09,
 'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.4041615334,
   'estimated_diameter_max': 0.9037326626},
  'meters': {'estimated_diameter_min': 404.1615334029,
   'estimated_diameter_max': 903.7326625794},
  'miles': {'estimated_diameter_min': 0.2511342562,
   'estimated_diameter_max': 0.5615532683},
  'feet': {'estimated_diameter_min': 1325.9893252496,
   'estimated_diameter_max': 2965.0022686971}},
 'is_potentially_hazardous_asteroid': True,
 'close_approach_data': [{'close_approach_date': '2029-04-13',
   'close_approach_date_full': '2029-Apr-13 21:46',
   'epoch_date_close_approach': 1870811160000,
   'relative_velocity': {'kilometers_per_second': '7.4